# Explore the effect of truncation size

In [9]:
# importing all required packages & notebook extensions at the start of the notebook
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import qiime2 as q2
from qiime2 import Visualization

%matplotlib inline

In [10]:
# get project root by finding .git folder
root = !git rev-parse --show-toplevel
root = root[0]

# assigning variables throughout the notebook
raw_data_dir = os.path.join(root, "data/raw")
data_dir = os.path.join(root, "data/processed")
vis_dir  = os.path.join(root, "results")

In [11]:
%%bash -s $raw_data_dir $data_dir $vis_dir
# Please do NOT modify this cell - here we copy the required data into
# your personal Jupyter workspace.

mkdir -p "$1" "$2" "$3"
chmod -R +rxw "$1" "$2" "$3"

In [12]:
# To inspect truncate & trim

qza = f"{data_dir}/sequences_trimmed.qzv"
a = !unzip -o $qza
digest = a[1].split('/')[0].replace("  inflating: ","")
fname_fwd = os.path.join(digest, "data/forward-seven-number-summaries.tsv")
fname_rev = os.path.join(digest, "data/reverse-seven-number-summaries.tsv")
fwd = pd.read_csv(fname_fwd, sep="\t", index_col=[0])
rev = pd.read_csv(fname_rev, sep="\t", index_col=[0])
!rm -r $digest

In [15]:
def choose_truncation_site(df, score_median=30, score_q1=25, rev=True):
    """
    Choose a truncation site `p` based on the following criteria
    1. all meidan q-score before `p` >= score_median
    2. average lower quatile (q1) q-score after `p` < score_q1
    3. if rev=False, take the smallest position that satisfies 1. and 2. (more stringent)
       otherwise, take the largest position that satisfies 1. and 2.
    """
    def _is_valid_truncation_site(df, col):
        # 4th row 50% quantile
        crit1 = np.all(df.iloc[4, :col] >= score_median)
        # 3rd row 25% quantile
        crit2 = df.iloc[3, col:].mean() < score_q1
        return crit1 and crit2
    
    it = range(len(df.columns))
    if rev:
        it = range(len(df.columns), 0, -1)
    for i in it:
        # return the first valid truncation site
        if _is_valid_truncation_site(df, i):
            return i


In [14]:
def choose_truncation_site_alt(df, score_median=30, score_q1=25):
    # Go over each position in the read (each column)
    for i in range(len(df.columns)):
        # If either condition breaks --> return truncation length
        if (df.iloc[3, i] < score_q1  # 3rd row (25% quantile) < score_q1
            or df.iloc[4, i] < score_median):   # 4th row (50% quantile) < score_q1
            return(i)

In [19]:
criteria = {}
criteria["strict"] = { "score_median": 30, "score_q1": 25 }
criteria["mild"]   = { "score_median": 25, "score_q1": 20 }

In [ ]:
strategy = "matchy"
for k, v in criteria.items():
    trunc_len_f = choose_truncation_site(fwd, **v)
    trunc_len_r = choose_truncation_site(rev, **v)
    ! qiime dada2 denoise-paired \
          --i-demultiplexed-seqs $data_dir/sequences_trimmed.qza \
          --p-trunc-len-f $trunc_len_f \
          --p-trunc-len-r $trunc_len_r \
          --p-n-threads 4 \
          --o-table $data_dir/table_$strategy_$k.qza \
          --o-representative-sequences $data_dir/rep-seqs_$strategy_$k.qza \
          --o-denoising-stats $data_dir/denoising-stats_$strategy_$k.qza

Saved FeatureTable[Frequency] to: /home/jovyan/project/alien/data/processed/table_.qza
Saved FeatureData[Sequence] to: /home/jovyan/project/alien/data/processed/rep-seqs_.qza
Saved SampleData[DADA2Stats] to: /home/jovyan/project/alien/data/processed/denoising-stats_.qza


In [ ]:
strategy = "anna"
for k, v in criteria.items():
    trunc_len_f = choose_truncation_site_alt(fwd, **v)
    trunc_len_r = choose_truncation_site_alt(rev, **v)
    ! qiime dada2 denoise-paired \
          --i-demultiplexed-seqs $data_dir/sequences_trimmed.qza \
          --p-trunc-len-f $trunc_len_f \
          --p-trunc-len-r $trunc_len_r \
          --p-n-threads 4 \
          --o-table $data_dir/table_${strategy}_${k}.qza \
          --o-representative-sequences $data_dir/rep-seqs_${strategy}_${k}.qza \
          --o-denoising-stats $data_dir/denoising-stats_${strategy}_${k}.qza